In [1]:
import os
import json
import gzip
import requests
import numpy as np
import pandas as pd

def download_nvd_feed(year):
    feed_url = f'https://nvd.nist.gov/feeds/json/cve/1.1/nvdcve-1.1-{year}.json.gz'
    response = requests.get(feed_url)

    with open(f'nvdcve-1.1-{year}.json.gz', 'wb') as f:
        f.write(response.content)

    return f'nvdcve-1.1-{year}.json.gz'

def extract_cve_from_item(item):
    if 'baseMetricV3' not in item['impact']:
        return None

    en_text = next(
        (desc['value'] for desc in item['cve']['description']['description_data']
         if desc['lang'] == 'en'), None)
    if en_text is None:
        return None

    en_text = en_text.replace('\n', ' ')
        #   Column                        Non-Null Count  Dtype  
        # ---  ------                        --------------  -----  
        #  0   publishedDate                 48427 non-null  object 
        #  1   lastModifiedDate              48427 non-null  object 
        #  2   CVE_ID                        48427 non-null  object 
        #  3   description                   48427 non-null  object 
        #  4   cvssV3_version                45926 non-null  float64
        #  5   cvssV3_vectorString           45926 non-null  object 
        #  6   cvssV3_attackVector           45926 non-null  object 
        #  7   cvssV3_attackComplexity       45926 non-null  object 
        #  8   cvssV3_privilegesRequired     45926 non-null  object 
        #  9   cvssV3_userInteraction        45926 non-null  object 
        #  10  cvssV3_scope                  45926 non-null  object 
        #  11  cvssV3_confidentialityImpact  45926 non-null  object 
        #  12  cvssV3_integrityImpact        45926 non-null  object 
        #  13  cvssV3_availabilityImpact     45926 non-null  object 
        #  14  cvssV3_baseScore              45926 non-null  float64
        #  15  cvssV3_baseSeverity           45926 non-null  object 
        #  16  V3_exploitabilityScore        45926 non-null  float64
        #  17  V3_impactScore                45926 non-null  float64
        #  18  nb_CWE                        48427 non-null  int64  
        #  19  CWE1                          48427 non-null  object 
        #  20  CWE2                          48427 non-null  object 
    return {
        'english_description': en_text,
        'base_score': item['impact']['baseMetricV3']['cvssV3']['baseScore'],
        'base_severity': item['impact']['baseMetricV3']['cvssV3']['baseSeverity'],
    }

def process_cve_data(years):
    all_cves = []
    skipped = 0
    processed = 0

    for year in years:
        filename = f'nvdcve-1.1-{year}.json.gz'

        if not os.path.exists(filename):
            print(f"Downloading data for year {year}...")
            filename = download_nvd_feed(year)
        else:
            print(f"File for year {year} already exists, using the existing file.")

        with gzip.open(filename, 'rt', encoding='utf-8') as f:
            nvd_data = json.load(f)

        print(f"Processing data for year {year}: {len(nvd_data['CVE_Items'])} total CVEs")

        year_cves = []
        for item in nvd_data['CVE_Items']:
            relevant_data = extract_cve_from_item(item)
            if relevant_data is None:
                skipped += 1
            else:
                year_cves.append(relevant_data)
                processed += 1

        all_cves.extend(year_cves)

    print(f"Processed CVEs: {processed}\nSkipped CVEs: {skipped}")
    return pd.DataFrame(all_cves)


In [3]:
years = [2022,2023,2024]
cvss_data = process_cve_data(years)

# Membuat kolom english_description diawali dengan tanda kutip 
cvss_data['english_description'] = cvss_data['english_description'].apply(lambda x: f'"{x}"')

# CLEANING : Ganti semua newline dengan spasi di kolom 'english_description'
cvss_data['english_description'] = cvss_data['english_description'].str.replace(r'[\n\r]+', ' ', regex=True)

cvss_data['predicted_score'] = None
cvss_data['predicted_severity'] = None

# Simpan data ke dalam file CSV jika diperlukan
cvss_data.to_csv('./data-for-real-test.csv', index=False)

File for year 2022 already exists, using the existing file.
Processing data for year 2022: 25287 total CVEs
File for year 2023 already exists, using the existing file.
Processing data for year 2023: 29080 total CVEs
File for year 2024 already exists, using the existing file.
Processing data for year 2024: 32975 total CVEs
Processed CVEs: 60401
Skipped CVEs: 26941


In [4]:
print(cvss_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60401 entries, 0 to 60400
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   english_description  60401 non-null  object 
 1   base_score           60401 non-null  float64
 2   base_severity        60401 non-null  object 
 3   predicted_score      0 non-null      object 
 4   predicted_severity   0 non-null      object 
dtypes: float64(1), object(4)
memory usage: 2.3+ MB
None


In [5]:
print(cvss_data.head)

<bound method NDFrame.head of                                      english_description  base_score  \
0      "Non-transparent sharing of branch predictor s...         6.5   
1      "Non-transparent sharing of branch predictor w...         6.5   
2      "Hardware debug modes and processor INIT setti...         6.8   
3      "Sensitive information accessible by physical ...         2.4   
4      "Insertion of Sensitive Information into Log F...         5.5   
...                                                  ...         ...   
60396  "Enterprise Cloud Database from Ragic does not...         9.8   
60397  "A vulnerability was found in code-projects Bl...         9.8   
60398  "A post-authentication SQL Injection vulnerabi...         8.8   
60399  "A maliciously crafted DWG file when parsed in...         7.8   
60400  "A maliciously crafted DWG file when parsed in...         7.8   

      base_severity predicted_score predicted_severity  
0            MEDIUM            None             